In [9]:
from collections import deque
import spot

# Reuse the structural LTLMutator class we refined in the previous step
# class LTLMutator:
#     def __init__(self, ap_list):
#         self.ap_set = set(ap_list)
#         self.aps = [spot.formula.ap(p) for p in self.ap_set]
#         self.unary_ops = {
#             spot.op_Not: spot.formula.Not, spot.op_X: spot.formula.X,
#             spot.op_F: spot.formula.F, spot.op_G: spot.formula.G
#         }
#         self.binary_ops = {
#             spot.op_U: spot.formula.U, spot.op_R: spot.formula.R, spot.op_W: spot.formula.W,
#             spot.op_And: lambda a, b: spot.formula.And([a, b]),
#             spot.op_Or: lambda a, b: spot.formula.Or([a, b])
#         }
#         self.rule3d_ops = {
#             'U': spot.formula.U, 'W': spot.formula.W,
#             '&': lambda a, b: spot.formula.And([a, b]), '|': lambda a, b: spot.formula.Or([a, b])
#         }

#     def mutate(self, phi):
#         if isinstance(phi, str):
#             phi = spot.formula(phi)
#         phi = phi.unabbreviate("ie")
#         mutations = {}
#         for mut_f in self._mutate_recursive(phi, is_top_level=True):
#             s = mut_f.to_str()
#             if s not in mutations:
#                 mutations[s] = mut_f
#         orig_s = phi.to_str()
#         if orig_s in mutations:
#             del mutations[orig_s]
#         return list(mutations.values())

#     def _mutate_recursive(self, f, is_top_level=True):
#         for op_func in self.unary_ops.values():
#             yield op_func(f)
            

#         if f.is_ff() or f in self.ap_set: yield spot.formula.tt()
#         if f.is_tt() or f in self.ap_set: yield spot.formula.ff()

#         if f in self.ap_set:
#             for p in self.aps:
#                 if p != f: yield p
        
#         if not is_top_level or f in self.ap_set or f.is_tt() or f.is_ff():
#             for p in self.aps: yield p

#         kind = f.kind()
#         if kind in self.unary_ops:
#             child = f[0]
#             for other_kind, op_func in self.unary_ops.items():
#                 if other_kind != kind: yield op_func(child)
#             yield child
#             for mutated_child in self._mutate_recursive(child, is_top_level=False):
#                 yield self.unary_ops[kind](mutated_child)
#             for p in self.aps:
#                 for op_func in self.rule3d_ops.values(): yield op_func(p, f)
                    
#         elif kind in self.binary_ops:
#             children = list(f)
#             if len(children) >= 2:
#                 phi_1 = children[0]
#                 phi_2 = spot.formula.And(children[1:]) if kind == spot.op_And else (spot.formula.Or(children[1:]) if kind == spot.op_Or else children[1])
                    
#                 for other_kind, op_func in self.binary_ops.items():
#                     if other_kind != kind: yield op_func(phi_1, phi_2)
#                 yield phi_1
#                 yield phi_2
#                 for mutated_left in self._mutate_recursive(phi_1, is_top_level=False):
#                     yield self.binary_ops[kind](mutated_left, phi_2)
#                 for mutated_right in self._mutate_recursive(phi_2, is_top_level=False):
#                     yield self.binary_ops[kind](phi_1, mutated_right)
class LTLMutator:
    def __init__(self, ap_list):
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        
        self.unary_ops = {
            spot.op_Not: spot.formula.Not,
            spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F,
            spot.op_G: spot.formula.G
        }
        
        self.binary_ops = {
            spot.op_U: spot.formula.U,
            spot.op_R: spot.formula.R,
            spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        
        self.rule3d_ops = {
            'U': spot.formula.U,
            'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]),
            '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        if isinstance(phi, str):
            phi = spot.formula(phi)
            
        phi = phi.unabbreviate("ie")
        mutations = {}
        
        for mut_f in self._mutate_recursive(phi):
            s = mut_f.to_str()
            if s not in mutations:
                mutations[s] = mut_f
                
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
            
        return list(mutations.values())

    # def _mutate_recursive(self, f):
    #     # --- GENERAL CASES ---
    #     for op_func in self.unary_ops.values():
    #         yield op_func(f)
            
    #     # yield spot.formula.tt()
    #     # yield spot.formula.ff()
    #     # for p in self.aps:
    #     #     yield p
            
    #     kind = f.kind()
        
    #     # # --- BASE CASES ---
    #     # if f.is_tt() or f.is_ff():
    #     #     yield spot.formula.ff()
    #     # if f.is_ff():
    #     #     yield spot.formula.tt()
            
    #     if f in self.ap_set:
    #         for p in self.aps:
    #             if p != f:
    #                 yield p
                    
    #     # --- INDUCTIVE CASES ---
    #     if kind in self.unary_ops:
    #         child = f[0]
            
    #         # 3(a) Change unary operator
    #         for other_kind, op_func in self.unary_ops.items():
    #             if other_kind != kind:
    #                 yield op_func(child)
                    
    #         # 3(b) Drop operator
    #         yield child
            
    #         # 3(c) Mutate child -> CRITICAL FIX: Added 'yield from'
    #         for mutated_child in self._mutate_recursive(child):
    #             yield self.unary_ops[kind](mutated_child)
                
    #         # 3(d) Append binary operator
    #         for p in self.aps:
    #             for op_func in self.rule3d_ops.values():
    #                 yield op_func(p, f)
                    
    #     elif kind in self.binary_ops:
    #         children = list(f)
            
    #         if len(children) >= 2:
    #             phi_1 = children[0]
    #             if len(children) > 2:
    #                 if kind == spot.op_And:
    #                     phi_2 = spot.formula.And(children[1:])
    #                 elif kind == spot.op_Or:
    #                     phi_2 = spot.formula.Or(children[1:])
    #                 else:
    #                     phi_2 = children[1]
    #             else:
    #                 phi_2 = children[1]
                    
    #             # 4(a) Change binary operator
    #             for other_kind, op_func in self.binary_ops.items():
    #                 if other_kind != kind:
    #                     yield op_func(phi_1, phi_2)
                        
    #             # 4(b) Keep one child
    #             yield phi_1
    #             yield phi_2
                
    #             # 4(c) Mutate left child -> CRITICAL FIX: Added 'yield from'
    #             for mutated_left in self._mutate_recursive(phi_1):
    #                 yield self.binary_ops[kind](mutated_left, phi_2)
                    
    #             # 4(d) Mutate right child -> CRITICAL FIX: Added 'yield from'
    #             for mutated_right in self._mutate_recursive(phi_2):
    #                 yield self.binary_ops[kind](phi_1, mutated_right)
                    
    def _mutate_recursive(self, f):
        # --- GENERAL CASES (Strictly restricted based on your structural goal) ---
        # Rule 5: Wrap the current sub-formula in a unary operator
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        kind = f.kind()
        
        # --- BASE CASES / LEAF MUTATIONS ---
        # Rule 1 & Rule 6 (Moved here so they only mutate leaves, not the whole tree)
        # if f.is_tt():
        #     yield spot.formula.ff()
            
        # if f.is_ff():
        #     yield spot.formula.tt()
            

        if f in self.ap_set:
            # Rule 2: Swap APs
            for p in self.aps:
                if p != f:
                    yield p
            # Rule 6 for APs: allow mapping an AP to other APs (handled above)
        
        # If we are at the top level of a complex formula, we block Rule 6 
        # from replacing the entire tree with a single 'p' or 'true'.
        # if not is_top_level or f in self.ap_set or f.is_tt() or f.is_ff():
        #     for p in self.aps:
        #         yield p

        # --- INDUCTIVE CASES ---
        if kind in self.unary_ops:
            child = f[0]
            
            # 3(a) Change unary operator
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind:
                    yield op_func(child)
                    
            # 3(b) Drop operator
            yield child
            
            # 3(c) Mutate child
            for mutated_child in self._mutate_recursive(child):
                yield self.unary_ops[kind](mutated_child)
                
            # 3(d) Append binary operator
            for p in self.aps:
                for op_func in self.rule3d_ops.values():
                    yield op_func(p, f)

            # 3(e) Swap adjacent unary operators: Op1(Op2(x)) -> Op2(Op1(x))
            if child.kind() in self.unary_ops:
                grandchild = child[0]
                outer_func = self.unary_ops[kind]
                inner_func = self.unary_ops[child.kind()]
                yield inner_func(outer_func(grandchild))
                
        # if kind in self.unary_ops:
        
        #     child = f[0]
            
        #     # 3(a) Change unary operator
        #     for other_kind, op_func in self.unary_ops.items():
        #         if other_kind != kind:
        #             yield op_func(child)
                    
        #     # 3(b) Drop operator
        #     yield child
            
        #     # 3(c) Mutate child
        #     for mutated_child in self._mutate_recursive(child):
        #         yield self.unary_ops[kind](mutated_child)
                
        #     # 3(d) Append binary operator
        #     for p in self.aps:
        #         for op_func in self.rule3d_ops.values():
        #             yield op_func(p, f)
                    
        elif kind in self.binary_ops:
            children = list(f)
            
            if len(children) >= 2:
                phi_1 = children[0]
                phi_2 = children[1]
                if len(children) > 2:
                    if kind == spot.op_And:
                        phi_2 = spot.formula.And(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    
                    
                # 4(a) Change binary operator
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind:
                        yield op_func(phi_1, phi_2)
                        
                # 4(b) Keep one child
                yield phi_1
                yield phi_2
                
                # 4(c) Mutate left child 
                for mutated_left in self._mutate_recursive(phi_1):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                    
                # 4(d) Mutate right child 
                for mutated_right in self._mutate_recursive(phi_2):
                    yield self.binary_ops[kind](phi_1, mutated_right)

class LTLMutationDistance:
    def __init__(self, ap_list):
        """
        Initialize the distance calculator with the allowed atomic propositions.
        """
        self.mutator = LTLMutator(ap_list)

    def calculate_distance(self, source, target):
        """
        Calculates the minimum mutation distance from source formula to target formula.
        
        :param source: Starting LTL formula string (e.g., "G(p U q)")
        :param target: Destination LTL formula string (e.g., "p U !q")
        :return: (int, list) The distance, and the step-by-step path taken. Returns (inf, []) if unreachable.
        """
        src_formula = spot.formula(source).unabbreviate("ie")
        tgt_formula = spot.formula(target).unabbreviate("ie")
        
        src_str = src_formula.to_str()
        tgt_str = tgt_formula.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        # BFS Queue holds tuples of (current_formula, path_taken_as_list)
        queue = deque([(src_formula, [src_str])])
        
        # Visited set tracks string representations to prevent infinite loops
        visited = {src_str}
        
        while queue:
            current_formula, current_path = queue.popleft()
            
            # Generate all valid 1-point mutations from the current formula
            neighbors = self.mutator.mutate(current_formula)
            
            for neighbor in neighbors:
                neighbor_str = neighbor.to_str()
                
                if neighbor_str == tgt_str:
                    return len(current_path), current_path + [neighbor_str]
                    
                if neighbor_str not in visited:
                    visited.add(neighbor_str)
                    queue.append((neighbor, current_path + [neighbor_str]))
                    
        return float('inf'), [] # If no structural mutation path connects them

In [10]:
# 1. Define the pool of atomic propositions used in your system
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "GF(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: GF(p U q)
Total Unique Mutations Found: 45
------------------------------
01: !GF(p U q)
02: XGF(p U q)
03: FGF(p U q)
04: !F(p U q)
05: XF(p U q)
06: F(p U q)
07: G!F(p U q)
08: GXF(p U q)
09: G!(p U q)
10: GX(p U q)
11: G(p U q)
12: GF!(p U q)
13: GFX(p U q)
14: GFG(p U q)
15: GF(p R q)
16: GF(p W q)
17: GF(p & q)
18: GF(p | q)
19: GFp
20: GFq
21: GF(!p U q)
22: GF(Xp U q)
23: GF(Fp U q)
24: GF(Gp U q)
25: GF(p U !q)
26: GF(p U Xq)
27: GF(p U Fq)
28: GF(p U Gq)
29: G(q U F(p U q))
30: G(q W F(p U q))
31: G(q & F(p U q))
32: G(q | F(p U q))
33: G(p U F(p U q))
34: G(p W F(p U q))
35: G(p & F(p U q))
36: G(p | F(p U q))
37: q U GF(p U q)
38: q W GF(p U q)
39: q & GF(p U q)
40: q | GF(p U q)
41: p U GF(p U q)
42: p W GF(p U q)
43: p & GF(p U q)
44: p | GF(p U q)
45: FG(p U q)


In [21]:
# Define the AP pool
ap_pool = ['p', 'q']
dist_calculator = LTLMutationDistance(ap_pool)

# Define source and destination formulas
start = "GF(p U q)"
end = "p U !(q->p)"

distance, path = dist_calculator.calculate_distance(start, end)

print(f"Source: {start}")
print(f"Target: {end}")
print(f"Mutation Distance: {distance}")
print("Optimal Mutation Path:")
print(" |-> ".join(path))

Source: GF(p U q)
Target: p U !(q->p)
Mutation Distance: 5
Optimal Mutation Path:
GF(p U q) |-> !F(p U q) |-> p U q |-> p U !q |-> p U (p | !q) |-> p U !(p | !q)


In [13]:
f_node = spot.formula("!GF(p U q)")
f_node.to_str()

'!GF(p U q)'

In [ ]:
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "!GF(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: GF(p U q)
Total Unique Mutations Found: 44
------------------------------
01: !GF(p U q)
02: XGF(p U q)
03: FGF(p U q)
04: !F(p U q)
05: XF(p U q)
06: F(p U q)
07: G!F(p U q)
08: GXF(p U q)
09: G!(p U q)
10: GX(p U q)
11: G(p U q)
12: GF!(p U q)
13: GFX(p U q)
14: GFG(p U q)
15: GF(p R q)
16: GF(p W q)
17: GF(p & q)
18: GF(p | q)
19: GFp
20: GFq
21: GF(!p U q)
22: GF(Xp U q)
23: GF(Fp U q)
24: GF(Gp U q)
25: GF(p U !q)
26: GF(p U Xq)
27: GF(p U Fq)
28: GF(p U Gq)
29: G(q U F(p U q))
30: G(q W F(p U q))
31: G(q & F(p U q))
32: G(q | F(p U q))
33: G(p U F(p U q))
34: G(p W F(p U q))
35: G(p & F(p U q))
36: G(p | F(p U q))
37: q U GF(p U q)
38: q W GF(p U q)
39: q & GF(p U q)
40: q | GF(p U q)
41: p U GF(p U q)
42: p W GF(p U q)
43: p & GF(p U q)
44: p | GF(p U q)


In [ ]:
from collections import deque
import spot

class LTLTree:
    """A pure, un-optimized Abstract Syntax Tree (AST) for LTL formulas."""
    def __init__(self, value, children=None):
        self.value = value  # e.g., 'G', 'F', '!', 'U', '&', 'p', 'q'
        self.children = children if children is not None else []

    @classmethod
    def from_spot(cls, f):
        """Recursively parses a Spot formula into a pure text AST."""
        kind = f.kind()
        
        if f.is_tt(): return cls("true")
        if f.is_ff(): return cls("false")
        
        # Check if Atomic Proposition
        if kind == spot.op_ap: 
            return cls(f.to_str())
        
        # Handle Unary Operators
        if kind in [spot.op_Not, spot.op_X, spot.op_F, spot.op_G]:
            op_str = {spot.op_Not: '!', spot.op_X: 'X', spot.op_F: 'F', spot.op_G: 'G'}[kind]
            return cls(op_str, [cls.from_spot(f[0])])
            
        # Handle Binary/N-ary Operators
        if kind in [spot.op_U, spot.op_R, spot.op_W, spot.op_And, spot.op_Or]:
            op_str = {spot.op_U: 'U', spot.op_R: 'R', spot.op_W: 'W', spot.op_And: '&', spot.op_Or: '|'}[kind]
            children = list(f)
            
            # FIX HERE: Manually fold flat multi-operand structures (e.g., [a, b, c]) 
            # into right-nested binary structures (e.g., (a & (b & c)))
            if len(children) > 2 and kind in [spot.op_And, spot.op_Or]:
                # Start from the last element and build upwards
                right_child = cls.from_spot(children[-1])
                for child in reversed(children[1:-1]):
                    right_child = cls(op_str, [cls.from_spot(child), right_child])
                return cls(op_str, [cls.from_spot(children[0]), right_child])
                
            return cls(op_str, [cls.from_spot(children[0]), cls.from_spot(children[1])])
            
        raise ValueError(f"Unsupported formula component: {f.to_str()}")

    def to_str(self):
        """Converts the tree back to a cleanly parenthesized string format."""
        if not self.children:
            return self.value
        if len(self.children) == 1:
            # Unary operators: e.g., !GF(p U q) -> !(G(F(p U q)))
            child_str = self.children[0].to_str()
            if len(self.value) > 1 or self.value.isalpha() or child_str.startswith('('):
                return f"{self.value}{child_str}"
            return f"{self.value}({child_str})"
        else:
            # Binary operators
            return f"({self.children[0].to_str()} {self.value} {self.children[1].to_str()})"

    def copy(self):
        return LTLTree(self.value, [c.copy() for c in self.children])


class PureLTLMutator:
    def __init__(self, ap_list):
        self.aps = ap_list
        self.unary_ops = ['!', 'X', 'F', 'G']
        self.binary_ops = ['U', 'R', 'W', '&', '|']

    def mutate(self, tree):
        mutations = set()
        for mut_tree in self._mutate_recursive(tree, is_top_level=True):
            mutations.add(mut_tree.to_str())
        orig_str = tree.to_str()
        if orig_str in mutations:
            mutations.remove(orig_str)
        return list(mutations)

    def _mutate_recursive(self, node, is_top_level=True):
        # --- GENERAL CASES ---
        # Rule 5: Wrap in unary operator
        for op in self.unary_ops:
            yield LTLTree(op, [node.copy()])
            
        # Rule 1 & 6: Constants/Leaves (Only at leaf level or structurally allowed)
        if node.value == "true": yield LTLTree("false")
        if node.value == "false": yield LTLTree("true")
        if node.value in ["false"] + self.aps: yield LTLTree("true")
        if node.value in ["true"] + self.aps: yield LTLTree("false")
        
        if node.value in self.aps:
            for p in self.aps:
                if p != node.value: yield LTLTree(p)
                
        if not is_top_level or node.value in (self.aps + ["true", "false"]):
            for p in self.aps:
                yield LTLTree(p)

        # --- INDUCTIVE CASES ---
        if node.value in self.unary_ops:
            child = node.children[0]
            
            # 3(a) Change unary operator
            for op in self.unary_ops:
                if op != node.value: yield LTLTree(op, [child.copy()])
            # 3(b) Drop operator
            yield child.copy()
            # 3(c) Mutate child
            for mut_child in self._mutate_recursive(child, is_top_level=False):
                yield LTLTree(node.value, [mut_child])
            # 3(d) Append binary operator
            for p in self.aps:
                for op in ['U', 'W', '&', '|']:
                    yield LTLTree(op, [LTLTree(p), node.copy()])

        elif node.value in self.binary_ops:
            left, right = node.children[0], node.children[1]
            
            # 4(a) Change binary operator
            for op in self.binary_ops:
                if op != node.value: yield LTLTree(op, [left.copy(), right.copy()])
            # 4(b) Keep one child
            yield left.copy()
            yield right.copy()
            # 4(c) Mutate left child
            for mut_left in self._mutate_recursive(left, is_top_level=False):
                yield LTLTree(node.value, [mut_left, right.copy()])
            # 4(d) Mutate right child
            for mut_right in self._mutate_recursive(right, is_top_level=False):
                yield LTLTree(node.value, [left.copy(), mut_right])


class PureLTLDistanceCalculator:
    def __init__(self, ap_list):
        self.mutator = PureLTLMutator(ap_list)

    def distance(self, start_formula, end_formula):
        # Normalize target string structure via the exact same syntax tree format
        src_tree = LTLTree.from_spot(spot.formula(start_formula))
        tgt_tree = LTLTree.from_spot(spot.formula(end_formula))
        
        src_str = src_tree.to_str()
        tgt_str = tgt_tree.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        queue = deque([(src_tree, [src_str])])
        visited = {src_str}
        
        while queue:
            curr_tree, path = queue.popleft()
            neighbors = self.mutator.mutate(curr_tree)
            
            for n_str in neighbors:
                if n_str == tgt_str:
                    return len(path), path + [n_str]
                if n_str not in visited:
                    visited.add(n_str)
                    queue.append((LTLTree.from_spot(spot.formula(n_str)), path + [n_str]))
                    
        return float('inf'), []
    

calc = PureLTLDistanceCalculator(['p', 'q'])
# Note: "q->p" maps syntactically to "!q | p" under standard unabbreviation tree formats
dist, path = calc.distance("GF(p U q)", "p U !( !q | p )")

print(f"Mutation Distance: {dist}")
print("Strict Structural Path:")
for step in path:
    print(f" -> {step}")

Mutation Distance: 5
Strict Structural Path:
 -> GF(p U q)
 -> GF(false U q)
 -> (p U GFq)
 -> (p U G!(q))
 -> (p U G(p | !(q)))
 -> (p U !(p | !(q)))


In [1]:
import spot

# 1. Parse your LTL formula
formula = spot.formula("G(a -> X b)")

# 2. Translate the formula into a Büchi Automaton
aut = formula.translate()

# 3. Find a satisfying run (a lasso)
run = aut.accepting_run()

if run:
    # This prints the handle and the cycle of the lasso
    print("Found a satisfying lasso:", run)
else:
    print("No satisfying lasso exists.")

Found a satisfying lasso: Prefix:
Cycle:
  0
  |  !a



<function spot._impl.formula_F>

In [4]:
import spot

def count_parameterized_k_lassos(formula_str, k):
    """
    Counts the number of satisfying k-lassos for a given LTL formula.
    k is a natural number parameter limiting the total steps (prefix + loop).
    """
    if k < 1:
        raise ValueError("k must be a natural number greater than 0")

    # 1. Translate the formula to an automaton
    f = spot.formula(formula_str)
    aut = f.translate()
    
    assert aut.is_existential(), "Automaton must be existential/alternating-free"
    
    start_state = aut.get_init_state_number()
    lasso_count = 0
    visited_lassos = set()

    # 2. Bounded DFS exploration
    def dfs(current_state, path, visited_states):
        nonlocal lasso_count
        
        # If the current path length exceeds k, we stop exploring this branch
        if len(path) > k:
            return

        # Check for a loop closure
        if current_state in visited_states:
            loop_start_idx = path.index(current_state)
            prefix = tuple(path[:loop_start_idx])
            loop = tuple(path[loop_start_idx:])
            
            # Total distinct transitions/steps in the lasso = len(prefix) + len(loop)
            # Since path contains the sequence of states up to the repeat, len(path) is our size
            if len(path) <= k:
                lasso_id = (prefix, loop)
                if lasso_id not in visited_lassos:
                    # Verify if the loop satisfies the Büchi acceptance condition
                    if is_loop_accepting(aut, path[loop_start_idx:]):
                        visited_lassos.add(lasso_id)
                        lasso_count += 1
            return

        # Traverse outgoing transitions
        for edge in aut.out(current_state):
            next_state = edge.dst
            
            new_visited = visited_states.copy()
            new_visited.add(current_state)
            
            dfs(next_state, path + [current_state], new_visited)

    def is_loop_accepting(automaton, loop_states):
        # Fix: Call spot.mark_t() directly to initialize an empty acceptance mark
        acc_sets = spot.mark_t() 
        
        for i in range(len(loop_states)):
            src = loop_states[i]
            dst = loop_states[(i + 1) % len(loop_states)]
            
            for edge in automaton.out(src):
                if edge.dst == dst:
                    acc_sets |= edge.acc
                    break
                    
        return automaton.get_acceptance().accepting(acc_sets)

    # Execute search
    dfs(start_state, [], set())
    return lasso_count

# --- Testing the Parameter ---
formula = "F(a) & G(b)"  # Eventually 'a', and 'b' always holds

for parameter_k in range(1, 6):
    count = count_parameterized_k_lassos(formula, k=parameter_k)
    print(f"For k = {parameter_k}, total satisfying lassos: {count}")

For k = 1, total satisfying lassos: 0
For k = 2, total satisfying lassos: 1
For k = 3, total satisfying lassos: 1
For k = 4, total satisfying lassos: 1
For k = 5, total satisfying lassos: 1


In [10]:
import spot

def count_lasso_bases(formula_str, k):
    """
    Approximates the number of bases w = s0...sk of length k+1
    that can form a lasso trace satisfying the given LTL formula.
    """
    f = spot.formula(formula_str)
    
    # Translate the formula into a deterministic monitor/Büchi automaton
    aut = spot.translate(f, 'ba', 'deterministic')
    
    # Dictionary to keep track of reachable states: {state_id: count_of_paths}
    current_states = {int(aut.get_init_state_number()): 1}
    
    # Track paths of length k (which translates to k+1 states)
    for step in range(k + 1):
        next_states = {}
        for state, path_count in current_states.items():
            for edge in aut.out(state):
                cond = edge.cond
                
                # If the edge condition is false, it's dead/unreachable
                if cond == spot.formula_ff():
                    continue
                
                next_state = int(edge.dst)
                next_states[next_state] = next_states.get(next_state, 0) + path_count
                    
        current_states = next_states

    # Total valid prefixes surviving up to bound k
    return sum(current_states.values())

def semantic_similarity(phi_str, psi_str, k):
    """
    Computes Semantic(phi, psi) = #Approx(phi & psi, k) / #Approx(phi | psi, k)
    """
    and_formula = f"({phi_str}) & ({psi_str})"
    or_formula = f"({phi_str}) | ({psi_str})"
    
    # To find the true set of APs safely, translate the combined formula to an automaton
    # and call .ap() on the automaton object, which returns a list of spot.formula objects
    combined_aut = spot.translate(spot.formula(or_formula), 'ba', 'deterministic')
    variables = set([ap.to_str() for ap in combined_aut.ap()])
    
    # Calculate counts
    num_intersection = count_lasso_bases(and_formula, k)
    num_union = count_lasso_bases(or_formula, k)
    
    print(f"Universal variables detected: {variables}")
    print(f"#Approx(φ ∧ ψ, k={k}) = {num_intersection}")
    print(f"#Approx(φ ∨ ψ, k={k}) = {num_union}")
    
    if num_union == 0:
        return 0.0  # Avoid division by zero
        
    return num_intersection / num_union

# --- Example Usage ---
if __name__ == "__main__":
    phi = "G(p -> X(q))"
    psi = "G(p -> F(q))"
    bound_k = 10
    
    similarity = semantic_similarity(phi, psi, bound_k)
    print(f"Semantic Similarity: {similarity:.4f}")

Universal variables detected: {'p', 'q'}
#Approx(φ ∧ ψ, k=10) = 2048
#Approx(φ ∨ ψ, k=10) = 2048
Semantic Similarity: 1.0000


In [1]:
"""
ltl_semantic_similarity.py
===========================

Model-counting-based semantic similarity between two LTL formulas.

    Semantic(phi, psi) = #(phi ∧ psi, k) / #(phi ∨ psi, k)

where #(alpha, k) is the (possibly approximate) number of accepting
lasso traces of bound k for formula `alpha`.

This module provides two things:

1. `semantic_similarity(...)`  -- the metric itself. It is agnostic to
   *how* models are counted: you inject a `model_counter_func` hook.
   This is the function that satisfies the stated requirements.

2. `naive_lasso_model_counter(...)` -- a reference/example implementation
   of a `model_counter_func`, built on top of `spot`. It brute-force
   enumerates all lasso traces (prefix p, cycle c, p + c <= k) over the
   atomic propositions of the formula and checks acceptance by
   intersecting a hand-built "single word" Büchi automaton with the
   formula's automaton. This is exponential in k and in the number of
   atomic propositions -- it exists to make the module runnable/testable
   out of the box and to demonstrate spot usage, NOT as a production
   model counter. For real workloads, replace it with a proper
   (approximate) bounded model counter, e.g. one based on #SAT / ADD
   symbolic counting of the bounded unrolling of the formula's automaton.

Requires: `pip install spot` (or your platform's spot package -- spot is
usually distributed via conda-forge or apt, since it's a SWIG-wrapped
C++ library).
"""

from __future__ import annotations

import itertools
from typing import Callable, List, Sequence, Union

import spot

# An LTL formula can be passed in either as a raw string or as an
# already-parsed spot.formula object.
LTLFormula = Union[str, "spot.formula"]

# A model counter takes a (possibly compound) LTL formula and a bound k,
# and returns a model count -- exact or approximate, int or float.
ModelCounterFunc = Callable[[LTLFormula, int], Union[int, float]]


# --------------------------------------------------------------------------
# Core metric
# --------------------------------------------------------------------------

def _to_spot_formula(f: LTLFormula) -> "spot.formula":
    """Normalize a formula (string or spot.formula) into a spot.formula."""
    if isinstance(f, spot.formula):
        return f
    parsed = spot.formula(f)
    if parsed.is_ff() and str(f).strip().lower() not in ("0", "false", "f"):
        # spot.formula() returns a "false" formula on unparsable input in
        # some binding versions instead of raising -- guard against a
        # silent typo turning into "unsat" rather than an error.
        raise ValueError(f"Could not parse LTL formula: {f!r}")
    return parsed


def semantic_similarity(
    phi: LTLFormula,
    psi: LTLFormula,
    k: int,
    model_counter_func: ModelCounterFunc,
) -> float:
    """
    Compute the model-counting semantic similarity between two LTL formulas.

        Semantic(phi, psi) = #(phi ∧ psi, k) / #(phi ∨ psi, k)

    Parameters
    ----------
    phi, psi:
        LTL formulas, either as strings (parseable by spot, e.g. "F(a) & X b")
        or as `spot.formula` objects.
    k:
        Bound on lasso trace length (prefix + cycle) used for model counting.
        Must be a non-negative integer.
    model_counter_func:
        Callable ``model_counter_func(formula: spot.formula, k: int) -> number``
        that returns the (exact or approximate) number of accepting lasso
        traces of bound `k` for `formula`. This function is called exactly
        twice: once for `phi ∧ psi` and once for `phi ∨ psi`.

    Returns
    -------
    float
        A value in [0.0, 1.0]. Returns 0.0 if the conjunction is
        unsatisfiable (count == 0) under the given bound, and 0.0 (instead
        of raising ZeroDivisionError) if the disjunction's count is also 0.

    Raises
    ------
    ValueError
        If `k` is negative, a formula fails to parse, or the model counter
        returns a negative count.
    """
    if k < 0:
        raise ValueError(f"Bound k must be non-negative, got {k}")

    phi_f = _to_spot_formula(phi)
    psi_f = _to_spot_formula(psi)

    conjunction = spot.formula.And([phi_f, psi_f])
    disjunction = spot.formula.Or([phi_f, psi_f])

    count_and = model_counter_func(conjunction, k)
    count_or = model_counter_func(disjunction, k)

    if count_and < 0 or count_or < 0:
        raise ValueError(
            f"model_counter_func returned a negative count "
            f"(and={count_and}, or={count_or}); model counts cannot be negative."
        )

    # phi ∧ psi unsatisfiable under bound k -> similarity is 0 by definition.
    if count_and == 0:
        return 0.0

    # Guard against division by zero. Note: if count_and > 0 then count_or
    # must also be > 0 (since phi ∧ psi implies phi ∨ psi, so the model set
    # of the conjunction is a subset of the disjunction's). This check is
    # kept anyway as a defensive measure in case model_counter_func is an
    # *approximate* counter where that subset guarantee can be violated by
    # estimation noise.
    if count_or == 0:
        return 0.0

    similarity = count_and / count_or

    # Clamp for safety against floating point drift or an inconsistent
    # (approximate) counter reporting count_and slightly > count_or.
    return max(0.0, min(1.0, float(similarity)))


# --------------------------------------------------------------------------
# Reference / example model counter (naive, spot-based, exponential)
# --------------------------------------------------------------------------

def _atomic_props(aut: "spot.twa_graph") -> List["spot.formula"]:
    """Return the list of atomic propositions registered on an automaton."""
    return list(aut.ap())


def _valuation_bdd(aut: "spot.twa_graph", ap_list: Sequence["spot.formula"],
                    valuation: Sequence[bool]) -> "spot.bdd":
    """Build the BDD for a single truth assignment over ap_list."""
    bdd_dict = aut.get_dict()
    cond = spot.buddy.bddtrue
    for ap, truth in zip(ap_list, valuation):
        var = bdd_dict.varnum(ap)
        lit = spot.buddy.bdd_ithvar(var) if truth else spot.buddy.bdd_nithvar(var)
        cond &= lit
    return cond


def _build_word_automaton(
    bdd_dict, ap_list: Sequence["spot.formula"], sequence: Sequence[Sequence[bool]],
    prefix_len: int,
) -> "spot.twa_graph":
    """
    Build a deterministic Büchi automaton that accepts exactly one infinite
    lasso word: `sequence[0..prefix_len-1]` followed by an infinite
    repetition of `sequence[prefix_len:]`.
    """
    total = len(sequence)
    cycle_len = total - prefix_len
    assert cycle_len >= 1, "lasso must have a non-empty cycle"

    word_aut = spot.make_twa_graph(bdd_dict)
    word_aut.set_buchi()
    for ap in ap_list:
        word_aut.register_ap(ap)

    word_aut.new_states(total)
    word_aut.set_init_state(0)

    for i in range(total):
        nxt = i + 1 if i + 1 < total else prefix_len  # loop back into the cycle
        cond = _valuation_bdd(word_aut, ap_list, sequence[i])
        in_cycle = i >= prefix_len
        word_aut.new_edge(i, nxt, cond, [0] if in_cycle else [])

    return word_aut


def naive_lasso_model_counter(formula: LTLFormula, k: int) -> int:
    """
    Reference implementation of a `model_counter_func`: brute-force counts
    the number of distinct lasso traces (prefix p, cycle c, 1 <= p + c <= k,
    c >= 1) over the formula's atomic propositions that satisfy `formula`.

    WARNING: this is exponential in `k` and in the number of atomic
    propositions -- it is meant for small formulas / small k (testing,
    demos, unit tests), not for production-scale similarity computation.
    Swap in a proper symbolic / approximate #SAT-style counter for real use.

    Parameters
    ----------
    formula: LTL formula (string or spot.formula).
    k: bound on prefix_len + cycle_len.

    Returns
    -------
    int: number of accepting lasso traces of bound k.
    """
    if k < 0:
        raise ValueError(f"Bound k must be non-negative, got {k}")
    if k == 0:
        return 0  # no valid lasso (needs at least a cycle of length >= 1)

    f = _to_spot_formula(formula)
    formula_aut = spot.translate(f, "BA", "complete")
    ap_list = _atomic_props(formula_aut)
    bdd_dict = formula_aut.get_dict()

    n_ap = len(ap_list)
    all_valuations = list(itertools.product([False, True], repeat=n_ap)) if n_ap else [()]

    count = 0
    # Enumerate all (prefix_len, cycle_len) shapes with prefix_len + cycle_len <= k
    for total_len in range(1, k + 1):
        for prefix_len in range(0, total_len):
            cycle_len = total_len - prefix_len
            if cycle_len < 1:
                continue
            # Enumerate all letter sequences of this exact length.
            for sequence in itertools.product(all_valuations, repeat=total_len):
                word_aut = _build_word_automaton(bdd_dict, ap_list, sequence, prefix_len)
                product = spot.product(word_aut, formula_aut)
                if not product.is_empty():
                    count += 1

    return count


# --------------------------------------------------------------------------
# Demo / usage example
# --------------------------------------------------------------------------

if __name__ == "__main__":
    phi = "F(a)"
    psi = "F(a) & G(!b)"
    k = 3

    sim = semantic_similarity(phi, psi, k, naive_lasso_model_counter)
    print(f"Semantic({phi!r}, {psi!r}, k={k}) = {sim:.4f}")

    # Unsatisfiable conjunction -> 0.0
    phi2, psi2 = "a", "!a"
    sim2 = semantic_similarity(phi2, psi2, k, naive_lasso_model_counter)
    print(f"Semantic({phi2!r}, {psi2!r}, k={k}) = {sim2:.4f}")  # expect 0.0

    # Identical formulas -> 1.0
    sim3 = semantic_similarity(phi, phi, k, naive_lasso_model_counter)
    print(f"Semantic({phi!r}, {phi!r}, k={k}) = {sim3:.4f}")  # expect 1.0

Semantic('F(a)', 'F(a) & G(!b)', k=3) = 1.0000
Semantic('a', '!a', k=3) = 0.0000
Semantic('F(a)', 'F(a)', k=3) = 1.0000
